# IRBANK企業ID（EID）取得

`master.csv`の国内株式銘柄を対象に、IRBANK（irbank.net）の企業ID（EID）・URL・社名を取得し、`stock/irbank.csv`を更新する。

## ローカル（Jupyter）実行専用の理由

GitHub Actionsのランナーから`irbank.net`へアクセスすると、robots.txtで`Allow: /`とされている直URL（`https://irbank.net/{code}`）ですら一律403 Forbiddenが返ることを確認済み（User-Agent変更でも回避できない）。IRBANK側がGitHub ActionsのデータセンターIPレンジをブロックしていると判断し、プロキシ等でのブロック回避は行わない方針とした。そのため、このnotebookを自分のPC上でJupyterから実行する運用にしている。

## 旧Notebook（`past/`）からの改良点

- EID解決は直URL（`https://irbank.net/{code}`）のみ。旧実装にあった検索ページ（`/search?q=...`）へのフォールバックは、robots.txtで明示的にDisallowされているため廃止した。
- 失敗時はHTTPステータス等の理由を記録し、原因調査をしやすくした。
- 取得結果は`master.csv`に列を追加する形ではなく、別ファイル`stock/irbank.csv`に保存する（`master.csv`はJPX公式データから毎回まるごと再生成される「作り直せるデータ」、`stock/irbank.csv`はスクレイピングで積み上げる「積み上げるデータ」のため。詳細は`../CLAUDE.md`の「データファイルの分離方針」参照）。
- 既に`status=ok`で取得済みの銘柄は自動でスキップする（中断・再実行に対応）。50件処理するごとに逐次保存するため、Jupyterのカーネルを途中で止めても、それまでの結果は失われない。

## 実行後にすること

更新された`app/brain/data/stock/irbank.csv`を、データリポジトリ（`palmelo2nd/brain_data`）に自分でコミット・pushする（このnotebookは保存のみ行い、git操作はしない）。pushすると、アプリの「データ更新」タブ→「企業ID」→「現在の状態」に反映される。

In [1]:
import random
import re
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

## 設定

In [2]:
# データリポジトリのローカルパス（notebooks/ から見た相対パス。app/brain/code/stock/notebooks/ -> app/brain/data/stock/）
DATA_DIR = Path("../../../data/stock")
MASTER_CSV = DATA_DIR / "master.csv"
IRBANK_CSV = DATA_DIR / "irbank.csv"

IRBANK_ASSET_TYPE = "内国株式"  # アプリ側（js/app.js）の絞り込みと揃えている

BASE_BY_CODE = "https://irbank.net/{code}"
RESULTS_URL = "https://irbank.net/{eid}/results"

USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"
ACCEPT_HEADER = "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8"

SLEEP_SEC = 1.2  # 銘柄ごとの取得間隔（秒）。IRBANK側への負荷配慮
SAVE_EVERY = 50  # この件数処理するごとに逐次保存する

JST = timezone(timedelta(hours=9))
IRBANK_COLUMNS = ["code", "eid", "url", "name", "status", "updated_at"]

# テスト用：Noneなら全件、数値を入れるとその件数だけ処理して様子を見られる
TEST_LIMIT = None

## 関数定義

In [3]:
def make_session() -> requests.Session:
    session = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT, "Accept": ACCEPT_HEADER})
    return session


def fetch_html(url: str, session: requests.Session, timeout: float = 20.0) -> str:
    response = session.get(url, timeout=timeout)
    response.raise_for_status()
    response.encoding = response.apparent_encoding or "utf-8"
    return response.text


def polite_sleep(base: float) -> None:
    """次のリクエストまでジッター付きで待機する（IRBANK側への負荷配慮）。"""
    time.sleep(max(0.0, base + random.uniform(-0.5 * base, 0.5 * base)))


def extract_name_from_html(html: str) -> str | None:
    """企業ページのh1タグから、先頭の証券コード表記を除いた社名を取り出す。"""
    soup = BeautifulSoup(html, "html.parser")
    h1 = soup.find("h1")
    if not h1:
        return None
    text = h1.get_text(strip=True)
    text = re.sub(r"^[0-9A-Za-z]+\s*", "", text)
    return text or None


def resolve_eid(code: str, session: requests.Session) -> tuple[str | None, str]:
    """証券コードから企業ID（EID）を、直URL（https://irbank.net/{code}）から解決する。
    見つかれば (EID, "")、見つからなければ (None, 診断用の失敗理由) を返す。
    """
    try:
        html = fetch_html(BASE_BY_CODE.format(code=code), session)
        soup = BeautifulSoup(html, "html.parser")
        a = soup.select_one('a[href^="/E"][href*="/results"]') or soup.select_one('a[href^="/E"]')
        if a and a.get("href"):
            m = re.search(r"/(E\d+)", a["href"])
            if m:
                return m.group(1), ""
        return None, f"EIDリンクが見つからない（取得HTML長={len(html)}文字）"
    except requests.HTTPError as e:
        status = e.response.status_code if e.response is not None else "?"
        return None, f"HTTPエラー status={status}"
    except Exception as e:
        return None, f"{type(e).__name__}: {e}"


def load_existing(irbank_csv: Path) -> dict[str, dict]:
    """既存のirbank.csvをcode→行辞書のマップとして読み込む（無ければ空）。"""
    if not irbank_csv.exists():
        return {}
    df = pd.read_csv(irbank_csv, dtype=str).fillna("")
    return {str(row["code"]): row.to_dict() for _, row in df.iterrows()}


def save_irbank_csv(rows: dict[str, dict], irbank_csv: Path) -> None:
    irbank_csv.parent.mkdir(parents=True, exist_ok=True)
    df = pd.DataFrame(list(rows.values()), columns=IRBANK_COLUMNS)
    df = df.sort_values("code")
    df.to_csv(irbank_csv, index=False)

## 対象銘柄の読み込み

`master.csv`から`status=listed`かつ`asset_type=内国株式`の銘柄を対象にし、既に`stock/irbank.csv`で`status=ok`の銘柄は除外する。

In [4]:
master_df = pd.read_csv(MASTER_CSV, dtype=str).fillna("")
target_df = master_df[(master_df["status"] == "listed") & (master_df["asset_type"] == IRBANK_ASSET_TYPE)]
all_target_codes = target_df["code"].tolist()

existing = load_existing(IRBANK_CSV)
target_codes = [c for c in all_target_codes if not (existing.get(c, {}).get("status") == "ok" and existing.get(c, {}).get("eid"))]

if TEST_LIMIT is not None:
    target_codes = target_codes[:TEST_LIMIT]

print(f"対象（内国株式・listed）: {len(all_target_codes)}銘柄 / 取得済みを除いた処理対象: {len(target_codes)}銘柄")

対象（内国株式・listed）: 3716銘柄 / 取得済みを除いた処理対象: 3167銘柄


## 取得実行

1銘柄の失敗（例外含む）は記録するだけで処理を継続する。`SAVE_EVERY`件ごとに`stock/irbank.csv`へ逐次保存するので、途中でカーネルを止めてもそこまでの結果は残る（再実行すれば続きから再開する）。

In [5]:
session = make_session()
succeeded = 0
failed = []

for i, code in enumerate(target_codes):
    try:
        eid, reason = resolve_eid(code, session)
        now = datetime.now(JST).strftime("%Y-%m-%d %H:%M:%S")

        if not eid:
            existing[code] = {"code": code, "eid": "", "url": "", "name": "", "status": "not_found", "updated_at": now}
            print(f"[not_found] コード: {code}（{reason}）")
            failed.append(code)
        else:
            url = RESULTS_URL.format(eid=eid)
            polite_sleep(SLEEP_SEC)
            name = ""
            try:
                name = extract_name_from_html(fetch_html(url, session)) or ""
            except Exception:
                pass

            existing[code] = {"code": code, "eid": eid, "url": url, "name": name, "status": "ok", "updated_at": now}
            succeeded += 1
            print(f"[ok] コード: {code} / EID: {eid} / 社名: {name}")
    except Exception as e:
        now = datetime.now(JST).strftime("%Y-%m-%d %H:%M:%S")
        existing[code] = {"code": code, "eid": "", "url": "", "name": "", "status": "not_found", "updated_at": now}
        print(f"エラーが発生しました（コード: {code}）: {e}")
        failed.append(code)

    if i < len(target_codes) - 1:
        polite_sleep(SLEEP_SEC)

    if (i + 1) % SAVE_EVERY == 0:
        save_irbank_csv(existing, IRBANK_CSV)
        print(f"--- 途中経過を保存しました（{i + 1}/{len(target_codes)}件処理） ---")

save_irbank_csv(existing, IRBANK_CSV)
print(f"\n完了: 成功 {succeeded}件 / 未取得・失敗 {len(failed)}件（対象 {len(target_codes)}件中）")
if failed:
    print(f"未取得・失敗コード: {', '.join(failed)}")

[ok] コード: 1301 / EID: E00012 / 社名: 極洋
[ok] コード: 130A / EID: E39268 / 社名: Veritas In Silico
[ok] コード: 1332 / EID: E00014 / 社名: ニッスイ
[ok] コード: 1333 / EID: E00015 / 社名: Umios
[ok] コード: 135A / EID: E39382 / 社名: VRAIN Solution
[ok] コード: 1375 / EID: E00007 / 社名: ユキグニファクトリー
[ok] コード: 1376 / EID: E00004 / 社名: カネコ種苗
[ok] コード: 1377 / EID: E00006 / 社名: サカタのタネ
[ok] コード: 1379 / EID: E00008 / 社名: ホクト
[ok] コード: 137A / EID: E39349 / 社名: Cocolive
[ok] コード: 1380 / EID: E00344 / 社名: 秋川牧園
[ok] コード: 1381 / EID: E00009 / 社名: アクシーズ
[ok] コード: 1382 / EID: E00010 / 社名: ホーブ
[ok] コード: 1383 / EID: E25969 / 社名: ベルグアース
[ok] コード: 1384 / EID: E31220 / 社名: ホクリヨウ
[ok] コード: 138A / EID: E39377 / 社名: 光フードサービス
[ok] コード: 1401 / EID: E00323 / 社名: エムビーエス
[ok] コード: 1407 / EID: E00327 / 社名: ウエスト HD
[ok] コード: 1414 / EID: E00329 / 社名: ショーボンド HD
[ok] コード: 1417 / EID: E24558 / 社名: ミライト・ワン
[ok] コード: 1418 / EID: E24512 / 社名: インターライフ HD
[ok] コード: 1419 / EID: E27305 / 社名: タマホーム
[ok] コード: 141A / EID: E38525 / 社名: トライアル HD
[ok] コード: 1420 

## 保存後の確認

`stock/irbank.csv`の内容を確認したら、データリポジトリ（`app/brain/data`）の変更を自分でコミット・pushする。

In [6]:
pd.read_csv(IRBANK_CSV, dtype=str).fillna("")

,code,eid,url,name,status,updated_at
0,1301,E00012,https://irbank.net/E00012/results,極洋,ok,2026-08-12 09:17:58
1,130A,E39268,https://irbank.net/E39268/results,Veritas In Silico,ok,2026-08-12 09:18:02
2,1332,E00014,https://irbank.net/E00014/results,ニッスイ,ok,2026-08-12 09:18:05
3,1333,E00015,https://irbank.net/E00015/results,Umios,ok,2026-08-12 09:18:08
4,135A,E39382,https://irbank.net/E39382/results,VRAIN Solution,ok,2026-08-12 09:18:11
...,...,...,...,...,...,...
3714,9996,E02786,https://irbank.net/E02786/results,サトー商会,ok,2026-08-12 12:30:25
3715,9997,E03229,https://irbank.net/E03229/results,ベルーナ,ok,2026-08-12 12:30:29
3716,<<<<<<< HEAD,,,,,
3717,=======,,,,,
